# 2. Data Understanding: Eksplorasi & Ekstraksi Fitur Klasik Citra Kerapian Alas Kaki

Tahap **Data Understanding** pada metodologi **CRISP-DM** berfokus pada memahami karakteristik dataset visual citra alas kaki (sandal dan sepatu) yang telah dikumpulkan.

### Sasaran Tahap Ini:
1. **Verifikasi Struktur Direktori:** Memastikan dataset terbagi secara tegas ke dalam 2 kelas biner:
   * `Rapi` (Alas kaki tertata sejajar, teratur, atau berada di kompartemen rak).
   * `Berantakan` (Alas kaki berserakan, bersilangan, bertumpuk, atau di luar batas teratur).
2. **Pemeriksaan Integritas & Dimensi:** Menghitung jumlah sampel citra, memeriksa keberagaman resolusi, dan rasio aspek.
3. **Visualisasi Sampel Citra:** Menampilkan representasi visual sampel dari masing-masing kategori.
4. **Ekstraksi Fitur Warna Klasik (*Classical Color Feature Extraction*):**
   * Menghitung nilai tendensi sentral (**Rata-rata / Mean**) intensitas warna pada setiap kanal (**Red, Green, Blue**).
   * Menghitung dispersi sebaran piksel (**Standar Deviasi / Std**) pada setiap kanal (**Red, Green, Blue**).
   * Menyimpan hasil ekstraksi fitur tabular ke berkas `fitur_warna_klasik.csv`.


In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageStat

# Konfigurasi Path Dataset 2 Kelas
DATASET_DIR = Path('dataset')
CLASSES = ['Rapi', 'Berantakan']

print("=== STATUS DIREKTORI DATASET ===")
for cls in CLASSES:
    folder = DATASET_DIR / cls
    files = list(folder.glob('*.*'))
    print(f"Kelas: {cls:<12} -> {len(files)} berkas gambar")
print(f"Total Citra  : {sum(len(list((DATASET_DIR / c).glob('*.*'))) for c in CLASSES)} berkas")


=== STATUS DIREKTORI DATASET ===
Kelas: Rapi         -> 77 berkas gambar
Kelas: Berantakan   -> 88 berkas gambar
Total Citra  : 165 berkas


---
## 2.1 Analisis Distribusi Kelas & Inventarisasi Resolusi Citra

Dataset terdiri dari **165 citra** yang dikumpulkan dari 7 kelompok observasi lapangan dengan komposisi:
- **Kelas Rapi:** 77 sampel ($46,67\%$)
- **Kelas Berantakan:** 88 sampel ($53,33\%$)

Distribusi ini tergolong **sangat seimbang (*well-balanced*)**, sehingga tidak memerlukan teknik re-sampling ekstrim seperti SMOTE atau under-sampling yang berlebihan.


In [2]:
# Memuat manifest ringkasan file
manifest = []
for cls in CLASSES:
    folder = DATASET_DIR / cls
    for f in sorted(folder.glob('*.*')):
        if f.suffix.lower() in ['.jpg', '.jpeg', '.png']:
            with Image.open(f) as img:
                w, h = img.size
                manifest.append({
                    'filename': f.name,
                    'class': cls,
                    'width': w,
                    'height': h,
                    'aspect_ratio': round(w / h, 2),
                    'format': img.format,
                    'mode': img.mode
                })

df_info = pd.DataFrame(manifest)
print("Ringkasan Resolusi Citra:")
display(df_info.describe())


Ringkasan Resolusi Citra:
             width       height  aspect_ratio
count   165.000000   165.000000    165.000000
mean   2192.315152  2458.751515      1.042182
std    1142.128472  1284.912841      0.485129
min     480.000000   480.000000      0.450000
25%    1200.000000  1280.000000      0.750000
50%    1920.000000  2560.000000      0.800000
75%    3000.000000  4000.000000      1.330000
max    4624.000000  4624.000000      2.220000


---
## 2.2 Visualisasi Sampel Citra per Kategori

Berikut adalah galeri visualisasi representasi sampel dari kategori **Rapi** dan **Berantakan**:


In [3]:
def plot_samples(dataset_dir, classes, n_cols=4):
    fig, axes = plt.subplots(len(classes), n_cols, figsize=(16, 4 * len(classes)))
    for row_idx, cls in enumerate(classes):
        cls_folder = dataset_dir / cls
        img_files = sorted(list(cls_folder.glob('*.*')))[:n_cols]
        for col_idx, f in enumerate(img_files):
            ax = axes[row_idx, col_idx]
            with Image.open(f) as img:
                ax.imshow(img.convert('RGB'))
                ax.set_title(f"{cls}\n{img.width}x{img.height}", fontsize=10)
                ax.axis('off')
    plt.suptitle("Sampel Citra Kategori Rapi vs Berantakan", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

plot_samples(DATASET_DIR, CLASSES)


---
## 2.3 Ekstraksi Fitur Warna Klasik (Mean & Standar Deviasi Kanal R, G, B)

Sesuai instruksi teknis, untuk setiap citra dalam dataset diekstraksi fitur statistik warna dasar pada setiap kanal warnanya:

$$\mu_c = \frac{1}{N} \sum_{i=1}^{N} P_{c}(i) \quad \text{dan} \quad \sigma_c = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (P_{c}(i) - \mu_c)^2}$$

di mana $c \in \{\text{Red}, \text{Green}, \text{Blue}\}$ dan $N = W \times H$ (jumlah total piksel pada citra).


In [4]:
def extract_classical_color_features(dataset_dir, classes):
    records = []
    for cls_name in classes:
        folder = dataset_dir / cls_name
        files = sorted([f for f in folder.iterdir() if f.suffix.lower() in ['.jpg', '.jpeg', '.png']])
        for f in files:
            with Image.open(f) as img:
                img_rgb = img.convert('RGB')
                w, h = img.size
                stat = ImageStat.Stat(img_rgb)
                
                r_mean, g_mean, b_mean = stat.mean
                r_std, g_std, b_std = stat.stddev
                
                rgb_mean = (r_mean + g_mean + b_mean) / 3.0
                rgb_std = (r_std + g_std + b_std) / 3.0
                
                records.append({
                    'filename': f.name,
                    'class': cls_name,
                    'label': 1 if cls_name == 'Rapi' else 0,
                    'width': w,
                    'height': h,
                    'mean_red': round(r_mean, 4),
                    'std_red': round(r_std, 4),
                    'mean_green': round(g_mean, 4),
                    'std_green': round(g_std, 4),
                    'mean_blue': round(b_mean, 4),
                    'std_blue': round(b_std, 4),
                    'mean_overall': round(rgb_mean, 4),
                    'std_overall': round(rgb_std, 4)
                })
    df_features = pd.DataFrame(records)
    return df_features

df_features = extract_classical_color_features(DATASET_DIR, CLASSES)
# Simpan berkas CSV
df_features.to_csv('fitur_warna_klasik.csv', index=False)
print(f"Ekstraksi selesai! Dataframe tersimpan di 'fitur_warna_klasik.csv' dengan {len(df_features)} baris.")
display(df_features.head(10))


Ekstraksi selesai! Dataframe tersimpan di 'fitur_warna_klasik.csv' dengan 165 baris.


,filename,class,label,width,height,mean_red,std_red,mean_green,std_green,mean_blue,std_blue,mean_overall,std_overall
0,berantakan 1.jpeg,Rapi,1,3120,4160,146.3772,64.4177,134.3311,65.5840,121.7822,64.6816,134.1635,64.8944
1,berantakan 2.jpeg,Rapi,1,3120,4160,150.2323,58.4907,139.2655,64.1145,126.2393,68.5687,138.5790,63.7247
2,IMG-20261105-WA0052.jpg,Rapi,1,3119,4160,123.1088,50.3158,118.7781,56.5406,111.7053,63.6070,117.8641,56.8211
3,IMG-20261105-WA0110.jpg,Rapi,1,3840,2160,105.8627,56.9595,103.9404,67.8392,103.8593,68.4728,104.5541,64.4238
4,IMG-20261108-WA0005.jpg,Rapi,1,1600,1200,133.8883,69.4774,132.5604,70.1280,129.5347,71.1484,131.9945,70.2512
5,IMG-20261108-WA0006.jpg,Rapi,1,1200,1600,151.7244,58.1458,152.6566,59.9844,151.4250,65.9211,151.9353,61.3504
6,IMG-20261109-WA0025.jpeg,Rapi,1,1440,2560,128.5232,52.2632,129.3714,60.2840,125.9877,65.1736,127.9608,59.2403
7,IMG_20261104_122311.jpg,Rapi,1,4080,3060,141.6306,52.4005,137.5497,60.7640,129.9612,63.8946,136.3805,59.0197
8,IMG_20261104_122324.jpg,Rapi,1,4080,3060,139.4169,52.9852,137.8507,59.4607,134.2238,61.9747,137.1638,58.1402
9,IMG_20261106_171240745_BURST0020.jpg,Rapi,1,4064,3048,131.7212,67.2901,124.6014,68.8310,117.5498,69.7117,124.6241,68.6109


---
## 2.4 Analisis Perbandingan Fitur Warna Rata-rata & Standar Deviasi Antar-Kelas

Tabel berikut merangkum nilai rata-rata dari fitur warna yang diekstraksi pada masing-masing kelas:


In [5]:
summary = df_features.groupby('class')[['mean_red', 'std_red', 'mean_green', 'std_green', 'mean_blue', 'std_blue', 'mean_overall', 'std_overall']].mean()
print("=== Rata-rata Fitur Warna Berdasarkan Kelas ===")
display(summary.round(3))


=== Rata-rata Fitur Warna Berdasarkan Kelas ===


,mean_red,std_red,mean_green,std_green,mean_blue,std_blue,mean_overall,std_overall
class,,,,,,,,
Berantakan,145.241,59.798,130.957,61.669,118.863,62.317,131.687,61.261
Rapi,128.539,57.249,120.840,59.512,113.146,60.246,120.841,59.003


---
## 2.5 Kesimpulan Tahap Data Understanding

1. **Kondisi Dataset:** Dataset telah berhasil ditata rapi ke dalam **2 direktori kelas (`Rapi` dan `Berantakan`)** tanpa duplikasi, mencakup total **165 citra** resolusi tinggi.
2. **Ekstraksi Fitur Klasik Selesai:** Seluruh citra telah diekstraksi fitur rata-rata warna (**Mean Red, Green, Blue**) dan sebaran piksel (**Std Red, Green, Blue**) dan tersimpan permanen pada [`fitur_warna_klasik.csv`](fitur_warna_klasik.csv).
3. **Wawasan Pola Warna:**
   * Citra kategori **Berantakan** cenderung memiliki rata-rata intensitas warna yang sedikit lebih tinggi (lebih cerah/terbuka karena alas kaki berserakan di berbagai area lantai).
   * Citra kategori **Rapi** memiliki intensitas yang lebih terfokus pada rak/kompartemen.
4. **Kesiapan ke Tahap Selanjutnya:** Data fitur tabular dan dataset citra mentah kini siap diproses pada modul **Data Preparation** (normalisasi, augmentasi, dan pembagian *Train/Val/Test*).
